# Chapter 4 hands-on: the sampler and the golden set

**Week 0 foundations guide, Chapter 4. Guided exercise, 60 minutes.** By the end you can show,
with numbers you produced, why one ticket got two labels, and you can say how far a model's
labels can be trusted, measured on twenty tickets labelled by hand.

> **Meera's question.** "Revenue fell from Q1 to Q2, where did it go?"

The support tickets Farhan's team labels are complaints about that same Q2, so a count of them
is one way into Meera's question. Before a count of refund tickets reaches her, two things have
to be known: how a model picks the label it returns, and how often its label matches a person's.

Each TODO is either a lettered choice, written in the comment above a `__TODOn__` placeholder,
where you replace the placeholder with the code of the letter you pick, or a prediction you type
as a value before the cell that tests it runs. Run as shipped, the notebook stops at the first
placeholder with a `NameError`, which is where it is meant to start. The scores and the twenty
tickets are practice data, constructed for this exercise and labelled that way wherever they
appear.

The next cell finds the programme's helper, `scripts/c2kit.py`, by walking up from this folder,
and draws where this exercise sits among the guide's eight. Nothing is read from a data folder,
because the practice scores and tickets are typed into the cells that use them.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

import csv
import io
import math
import random

kit.side_by_side(
    kit.ladder(["Python", "SQL", "Numbers", "Language models", "Business problems", "pandas",
                "Excel", "GitHub"], lit=3, title="The guide's eight hands-on exercises", show=False),
    kit.vflow(["the call anatomy, redrawn", "scores in, probabilities out", "predict, then sample",
               "trace five draws by hand", "spread the picks, then validate them",
               "twenty tickets, two columns", "break it: one number for the whole set",
               "precision and recall per label", "the mini project's self-check"],
              title="Nine steps, 60 minutes", show=False),
)

## 1. The call anatomy, redrawn (5 minutes)

Before any code, redraw the chapter's picture on paper from memory: the six stages from the text
you send to the object that comes back, with a ring around the one stage that explains why the
same ticket got refund once and delivery the next time. Then run the cell below and compare it
with your drawing, stage by stage.

In [ ]:
kit.flow(["Prompt\ntext you send", "Tokens\npieces of text", "Model\nnext-token scores",
          "Sampling\ntemperature picks", "Reply\ntokens joined", "Response object\ncontent + usage"],
         lit=3, title="The call anatomy (Figure 18): each call is stateless, and sampling is the stage behind Q33")

If your ring sat on the model, look at what the model hands on: a score for every token that could
come next. The model's opinion is settled at that point, and the pick from those scores is the
sampling stage, which is where one ticket can become two labels.

## 2. Scores in, probabilities out (7 minutes)

The model has just read Q33's prompt, which asks for a label and never lists the labels, and it
has scored five candidate next tokens. The scores below are constructed for practice, shaped so
that at temperature 1 refund and delivery take Figure 19's 55 and 35 percent, and the last tenth
goes to other and to two labels the prompt never ruled out. A softmax turns scores into
probabilities: divide each score by the temperature, raise e to that power to get a weight, and
share the weights out so they add to 1. Temperature 0 cannot be divided by, so the function
handles it first with the rule the chapter states, the top-scoring token every time.

In [ ]:
PROMPT = 'Label this ticket: "My parcel is a week late and I want my money back."'
SCORES = {"refund": 2.40, "delivery": 1.95, "other": -0.22, "return": -0.22, "late": -0.91}
ALLOWED = ("refund", "delivery", "other")

print(PROMPT)
kit.table(["candidate next token", "score", "one of the allowed labels"],
          [(token, f"{score:.2f}", "yes" if token in ALLOWED else "no") for token, score in SCORES.items()],
          caption="Practice scores for one prompt, constructed for this exercise")

In [ ]:
def probabilities(scores, temperature):
    """Turn next-token scores into probabilities at one temperature."""
    if temperature == 0:
        top = max(scores, key=scores.get)
        return {token: (1.0 if token == top else 0.0) for token in scores}
    # TODO 1. Which expression is a token's weight before the weights are shared out?
    #   a) score / temperature
    #   b) math.exp(score / temperature)
    #   c) math.exp(score) / temperature
    #   d) math.exp(score * temperature)
    weights = {token: __TODO1__ for token, score in scores.items()}
    total = sum(weights.values())
    return {token: weight / total for token, weight in weights.items()}


at_1 = probabilities(SCORES, 1)
at_2 = probabilities(SCORES, 2)
kit.bars([(token, round(p * 100, 1)) for token, p in at_1.items()], fmt=lambda v: f"{v:.1f} percent",
         lit=(0,), title="Probabilities at temperature 1, from the practice scores")

In [ ]:
kit.check("the probabilities add to 1", abs(sum(at_1.values()) - 1) < 1e-9, f"{sum(at_1.values()):.6f}")
kit.check("every probability sits between 0 and 1", all(0 <= p <= 1 for p in at_1.values()),
          {token: round(p, 4) for token, p in at_1.items()})
kit.check("at temperature 1, refund and delivery take Figure 19's 55 and 35 percent",
          round(at_1["refund"], 2) == 0.55 and round(at_1["delivery"], 2) == 0.35,
          f'refund {at_1["refund"]:.4f}, delivery {at_1["delivery"]:.4f}')
kit.check("a higher temperature gives the top token a smaller share", at_2["refund"] < at_1["refund"],
          f'{at_1["refund"]:.4f} at temperature 1, {at_2["refund"]:.4f} at 2')

## 3. Predict, then sample (7 minutes)

The sampler draws a number u between 0 and 1, walks the probabilities in order with a running
total, and stops on the first token whose running total passes u. It draws 1,000 times with a
fixed seed, so every run of this notebook sees the same draws and your numbers match the
solution's. Write your two predictions in the next cell before you run the sampler.

In [ ]:
# TODO 2. Predict: at temperature 0, how many of the 1,000 picks will be refund? Type a whole number.
predict_refund_at_0 = __TODO2__

# TODO 3. Predict: at temperature 1, about how many of the 1,000 picks will be delivery?
predict_delivery_at_1 = __TODO3__
print(f"You predict {predict_refund_at_0} refund picks at temperature 0 and about "
      f"{predict_delivery_at_1} delivery picks at temperature 1.")

In [ ]:
def sample(scores, temperature, n=1000, seed=107):
    """Draw n picks from the scores at one temperature, with a fixed seed."""
    probs = probabilities(scores, temperature)
    rng = random.Random(seed)
    picks, draws = [], []
    for _ in range(n):
        u = rng.random()
        running, pick = 0.0, list(probs)[-1]
        for token, p in probs.items():
            running = running + p
            if u < running:
                pick = token
                break
        draws.append(u)
        picks.append(pick)
    return picks, draws


def count(picks):
    return {token: picks.count(token) for token in SCORES}


picks_0, _ = sample(SCORES, 0)
picks_1, draws_1 = sample(SCORES, 1)
counts_0, counts_1 = count(picks_0), count(picks_1)
kit.table(["token", "picks at temperature 0", "picks at temperature 1"],
          [(token, counts_0[token], counts_1[token]) for token in SCORES],
          caption="1,000 picks of the same ticket at each temperature, seed 107")
kit.columns(list(SCORES), [("temperature 0", [counts_0[token] for token in SCORES]),
                           ("temperature 1", [counts_1[token] for token in SCORES])],
            lit=(0,), title="The same scores, two picking rules (Figure 19, run 1,000 times)")

In [ ]:
kit.check("each run made 1,000 picks", len(picks_0) == len(picks_1) == 1000)
kit.check("your temperature 0 prediction matches the run", predict_refund_at_0 == counts_0["refund"],
          f'you said {predict_refund_at_0}, the run gave {counts_0["refund"]}')
kit.check("your temperature 1 prediction for delivery is within 40 of the run",
          abs(predict_delivery_at_1 - counts_1["delivery"]) <= 40,
          f'you said {predict_delivery_at_1}, the run gave {counts_1["delivery"]}')

## 4. Trace five draws by hand (6 minutes)

The next cell prints the band of u that each token owns at temperature 1, and the first five
draws the sampler made at that temperature. Fill the trace below it by hand: for each draw, find
the band that holds its u and write that token. The check then compares your trace with the
sampler's own.

In [ ]:
bands, low = [], 0.0
for token, p in at_1.items():
    bands.append((token, f"{low:.4f}", f"{low + p:.4f}"))
    low = low + p
kit.table(["token", "u from", "up to"], bands,
          caption="The bands at temperature 1: a draw picks the token whose band holds its u")
kit.table(["draw", "u"], [(n + 1, f"{u:.4f}") for n, u in enumerate(draws_1[:5])],
          caption="The first five draws at temperature 1")

In [ ]:
# TODO 4 to 8. For each draw, write the token whose band holds its u, as a string such as "other".
my_trace = [
    # (draw, u, the token it picks)
    (1, 0.2465, __TODO4__),
    (2, 0.5813, __TODO5__),
    (3, 0.9723, __TODO6__),
    (4, 0.6578, __TODO7__),
    (5, 0.4825, __TODO8__),
]
print("Your trace:", ", ".join(row[2] for row in my_trace))

In [ ]:
real_trace = [(n + 1, round(u, 4), pick) for n, (u, pick) in enumerate(zip(draws_1[:5], picks_1[:5]))]
kit.table(["draw", "u", "your token", "the sampler's token"],
          [(real[0], f"{real[1]:.4f}", mine[2], real[2]) for mine, real in zip(my_trace, real_trace)],
          caption="Your trace beside the sampler's")
kit.check("your trace has one row per draw", len(my_trace) == len(real_trace) == 5, len(my_trace))
kit.check("the u values in your trace are the sampler's",
          all(abs(mine[1] - real[1]) < 1e-4 for mine, real in zip(my_trace, real_trace)))
kit.check("your trace picks the sampler's token on all five draws",
          [row[2] for row in my_trace] == [row[2] for row in real_trace],
          ", ".join(row[2] for row in real_trace))

## 5. Spread the picks, then validate them (7 minutes)

The same ticket now runs at four temperatures, and every pick goes through a validator that keeps
only the three allowed labels. The chapter's point about temperature 0 is that it makes replies
far more repeatable, that vendors do not promise identical replies, and that a pipeline which
needs the same answer every time still checks each reply against the allowed set. Predict before
you run.

In [ ]:
# TODO 9. Predict: at temperature 0, how many of the 1,000 picks will fall outside the allowed set?
predict_outside_at_0 = __TODO9__


def valid(pick):
    """True when a pick is one of the allowed labels."""
    # TODO 10. Which test keeps a pick only when it is one of the three allowed labels?
    #   a) pick in SCORES
    #   b) pick != ""
    #   c) pick in ALLOWED
    #   d) len(pick) > 0 and pick.isalpha()
    return __TODO10__


print(f"You predict {predict_outside_at_0} picks outside the allowed set at temperature 0.")

In [ ]:
TEMPERATURES = [0, 0.5, 1, 2]
sweep = {}
for temperature in TEMPERATURES:
    picks, _ = sample(SCORES, temperature)
    sweep[temperature] = {"counts": count(picks), "outside": sum(1 for p in picks if not valid(p))}
kit.table(["temperature", *SCORES, "outside the allowed set"],
          [(temperature, *[sweep[temperature]["counts"][token] for token in SCORES], sweep[temperature]["outside"])
           for temperature in TEMPERATURES],
          caption="1,000 picks of the same ticket at each temperature, seed 107")
kit.line([str(temperature) for temperature in TEMPERATURES],
         [("refund", [sweep[temperature]["counts"]["refund"] for temperature in TEMPERATURES], "lit"),
          ("delivery", [sweep[temperature]["counts"]["delivery"] for temperature in TEMPERATURES], "accent"),
          ("outside the allowed set", [sweep[temperature]["outside"] for temperature in TEMPERATURES], "bad")],
         title="Picks out of 1,000 as the temperature rises: the top token loses share, invented labels gain it")

In [ ]:
outside = [sweep[temperature]["outside"] for temperature in TEMPERATURES]
refunds = [sweep[temperature]["counts"]["refund"] for temperature in TEMPERATURES]
kit.check("your prediction for temperature 0 matches the run", predict_outside_at_0 == outside[0],
          f"you said {predict_outside_at_0}, the run gave {outside[0]}")
kit.check("temperature 0 takes the top token on all 1,000 picks", refunds[0] == 1000, refunds[0])
kit.check("the validator keeps the three allowed labels and rejects the two invented ones",
          [valid(token) for token in SCORES] == [True, True, True, False, False])
kit.check("picks outside the allowed set rise with the temperature",
          outside == sorted(outside) and outside[-1] > outside[0], outside)
kit.check("the top token's share falls as the temperature rises", refunds == sorted(refunds, reverse=True),
          refunds)

**What the sweep says.** Temperature 0 made this ticket's label repeatable, and the label it
repeats is refund. The hand rule in the next step labels the same complaint delivery (it is ticket
T04), so temperature 0 repeated an answer a person would call wrong: the scores were already the
model's opinion, and the dial only decides how the pick is made from them. The chapter's first fix
is in the prompt, the label list with an example each, and the validator stays on at every
temperature.

In [ ]:
kit.flow(["Scores\nthe model's opinion", "Softmax at a temperature\nprobabilities",
          "Sample\none pick per call", "Validate\nonly allowed labels pass"],
         kinds=["plain", "plain", "plain", "good"], title="What the sampler half established")
kit.table(["What you can now say", "The evidence from this run"],
          [("Temperature 0 takes the top token every time", f"refund on {refunds[0]:,} of 1,000 picks"),
           ("Higher temperatures spread the picks",
            f"refund falls from {refunds[0]:,} to {refunds[-1]} picks between temperature 0 and 2"),
           ("Invented labels appear once the picks spread, so the validator stays on",
            f"{outside[1]} at 0.5, {outside[2]} at 1 and {outside[3]} at 2, of 1,000 picks each"),
           ("Repeatable is a different property from right", "the hand rule labels this ticket delivery")],
          caption="The sampler half, in four lines")

## 6. Twenty tickets, two columns (6 minutes)

The golden set is Figure 21's loop run on twenty tickets: labels written by hand first, the
model's labels beside them, and a comparison that produces a rate you can state. The twenty
tickets below are practice data, written for this exercise in the style of the Q2 complaints
Farhan's team labels, and the practice model column stands in for the labels your own assistant
will give. The hand labels follow one rule, written down before any ticket was labelled:

- **delivery** when the complaint is about the parcel reaching the customer: late, missing,
  damaged on the way or taken to the wrong door, even when the customer asks for the money back;
- **refund** when the complaint is about money itself: a double charge, an overcharge, or a
  refund owed and never received;
- **other** for everything else, such as a question or a change to an account.

In [ ]:
kit.flow(["Sample\nat random, not the easy ones", "Label by hand\nyour judgment, written",
          "Run the prompt\nthe same tickets", "Compare\nmatch or mismatch", "Agreement rate\nper label too"],
         lit=4, title="The golden set (Figure 21), run here on twenty practice tickets")

TICKETS = [
    # (ticket, text, your hand label, the practice model's label): practice data for this exercise
    ("T01", "I was charged twice for one order, please return one payment.", "refund", "refund"),
    ("T02", "My parcel shows delivered but nothing came to my door.", "delivery", "delivery"),
    ("T03", "How do I change the phone number on my account?", "other", "other"),
    ("T04", "My parcel is a week late and I want my money back.", "delivery", "refund"),
    ("T05", "I cancelled on Monday and the refund has still not reached my card.", "refund", "refund"),
    ("T06", "The box arrived crushed and the glass inside is broken.", "delivery", "delivery"),
    ("T07", "Do you sell the same kurta in a larger size?", "other", "other"),
    ("T08", "The courier marked my address as not found although it is correct.", "delivery", "delivery"),
    ("T09", "You charged Rs 1,200 for an item listed at Rs 950.", "refund", "refund"),
    ("T10", "My order never arrived, so please refund me or send it again.", "delivery", "refund"),
    ("T11", "Please change the delivery address on my next order.", "other", "delivery"),
    ("T12", "The refund for my returned shoes shows only half the amount.", "refund", "refund"),
    ("T13", "Tracking has not moved for five days.", "delivery", "delivery"),
    ("T14", "Can I add a gift note to an order I have already placed?", "other", "other"),
    ("T15", "The courier left my parcel in the rain and it is ruined, so I want my money back.",
     "delivery", "refund"),
    ("T16", "The discount code was not applied and I paid the full price.", "refund", "refund"),
    ("T17", "The parcel went to my neighbour's flat instead of mine.", "delivery", "delivery"),
    ("T18", "Is cash on delivery available for my pin code?", "other", "other"),
    ("T19", "My order came three days after the promised date.", "delivery", "delivery"),
    ("T20", "I was billed for a membership I never signed up for.", "refund", "refund"),
]
kit.table(["ticket", "text", "your hand label", "practice model"], TICKETS,
          caption="Practice data, constructed for this exercise: twenty tickets, hand labels to the rule above, and a practice model column")

Before the loop runs, count by eye: on how many of the twenty rows do the two columns carry the
same label? Write it in the next cell, then pick the update that counts an agreement.

In [ ]:
# TODO 11. Predict: on how many of the 20 tickets do the two columns agree? Type a whole number.
predict_matches = __TODO11__

matches = 0
for ticket_id, text, hand, model in TICKETS:
    if hand == model:
        # TODO 12. Which update counts this ticket as an agreement?
        #   a) matches = matches + 1
        #   b) matches = 1
        #   c) matches = matches + len(hand)
        #   d) matches = matches + matches
        __TODO12__
agreement = matches / len(TICKETS)
print(f"The practice model agrees with the hand labels on {matches} of {len(TICKETS)} tickets, {agreement:.0%}.")

In [ ]:
kit.check("the golden set holds 20 tickets", len(TICKETS) == 20, len(TICKETS))
kit.check("agreement is a share between 0 and 1", 0 <= agreement <= 1, f"{agreement:.2f}")
kit.check("your count by eye matches the loop", predict_matches == matches,
          f"you said {predict_matches}, the loop counted {matches}")

## 7. Break it on purpose: one number for the whole set (8 minutes)

The rate the loop just printed sits close to the chapter's "84 percent agreement on 30
hand-labelled questions", and the tempting next move is to call the labels trustworthy and count
the complaints from the model's column. Predict before you run: taking the practice model's
column as it stands, which label will the count report as the largest complaint?

**The plausible wrong answer.** The hurried analyst's cell, exactly as it would be written:

In [ ]:
# TODO 13. Predict: which label will the hurried count report as the largest? Type it as a string.
predict_top_label = __TODO13__

hurried = {label: 0 for label in ALLOWED}
for ticket_id, text, hand, model in TICKETS:
    hurried[model] = hurried[model] + 1
top = max(hurried, key=hurried.get)
print(f"Agreement {matches} of {len(TICKETS)} ({agreement:.0%}), so the labels are used as they stand.")
print(f"Largest complaint: {top}, {hurried[top]} of {len(TICKETS)} tickets ({hurried[top] / len(TICKETS):.0%}).")

**Why it is wrong.** The 80 percent is true, and it says nothing about where the four misses
went. If most of them landed in one label, that label's count runs high and the ranking can flip,
so Meera would hear about the wrong complaint for her Q2. The check below counts the same twenty
tickets from both columns before anything is ranked.

In [ ]:
hand_counts = {label: 0 for label in ALLOWED}
for ticket_id, text, hand, model in TICKETS:
    hand_counts[hand] = hand_counts[hand] + 1
hand_top = max(hand_counts, key=hand_counts.get)
kit.columns(list(ALLOWED), [("your hand labels", [hand_counts[label] for label in ALLOWED]),
                            ("practice model", [hurried[label] for label in ALLOWED])],
            lit=(0,), title="The same twenty tickets, counted from each column")
kit.check("your prediction of the hurried count's largest label matches the run", predict_top_label == top,
          f"you said {predict_top_label!r}, the run gave {top!r}")
kit.check("the hand labels rank a different label first", hand_top != top,
          f"hand labels: {hand_top}; model column: {top}")
kit.check("the model column carries more refund tickets than the hand labels",
          hurried["refund"] > hand_counts["refund"], f'{hurried["refund"]} against {hand_counts["refund"]}')

## 8. Precision and recall per label (8 minutes)

**The fix.** Report the result per label, as the chapter says, because one accuracy number hides
the label a model over-predicts. Precision asks, of the tickets the model gave a label, what share
you gave it too; recall asks, of the tickets you gave a label, what share the model found.

In [ ]:
model_counts = hurried
both = {label: 0 for label in ALLOWED}
for ticket_id, text, hand, model in TICKETS:
    if hand == model:
        both[hand] = both[hand] + 1

precision, recall = {}, {}
for label in ALLOWED:
    # TODO 14. Which expression is the precision of a label?
    #   a) both[label] / hand_counts[label]
    #   b) model_counts[label] / hand_counts[label]
    #   c) both[label] / len(TICKETS)
    #   d) both[label] / model_counts[label]
    precision[label] = __TODO14__
    # TODO 15. And which is its recall?
    #   a) both[label] / model_counts[label]
    #   b) both[label] / hand_counts[label]
    #   c) hand_counts[label] / model_counts[label]
    #   d) (both[label] + 1) / (hand_counts[label] + 1)
    recall[label] = __TODO15__

over = [label for label in ALLOWED if model_counts[label] > hand_counts[label]]
kit.table(["label", "you gave it", "the model gave it", "both agreed", "precision", "recall"],
          [(label, hand_counts[label], model_counts[label], both[label], f"{precision[label]:.2f}",
            f"{recall[label]:.2f}") for label in ALLOWED],
          caption="The practice golden set, per label")
kit.matrix(list(ALLOWED), list(ALLOWED),
           [[str(sum(1 for _, _, hand, model in TICKETS if hand == row and model == column))
             for column in ALLOWED] for row in ALLOWED],
           title="Rows are your hand labels and columns the practice model's")

In [ ]:
kit.check("each label's precision and recall sit between 0 and 1",
          all(0 <= precision[label] <= 1 and 0 <= recall[label] <= 1 for label in ALLOWED))
kit.check("the agreements per label add back to the loop's count", sum(both.values()) == matches, both)
kit.check("the model over-predicts exactly one label, refund", over == ["refund"], over)
kit.check("refund has the lowest precision and a recall of 1",
          min(precision, key=precision.get) == "refund" and recall["refund"] == 1,
          f'precision {precision["refund"]:.2f}, recall {recall["refund"]:.2f}')

**What the table says.** Refund's precision of 0.67 means a third of the model's refund labels
belong elsewhere, and its recall of 1.00 means it never missed a real refund, so the model
over-predicts refund. Delivery's recall of 0.67 shows where those tickets came from. Until the
prompt is fixed and measured again, a Q2 complaint split comes from the hand labels, which put
delivery first. The misses below are the README's raw material, and the three filed under the
over-predicted label are the three worst, since they are what lifted refund to the top.

In [ ]:
misses = [(ticket_id, text, hand, model) for ticket_id, text, hand, model in TICKETS if hand != model]
worst = [miss for miss in misses if miss[3] in over]
kit.table(["ticket", "text", "your hand label", "practice model"], misses,
          caption="Every miss in the practice golden set")
kit.check("the misses and the agreements add up to twenty", len(misses) + matches == len(TICKETS),
          f"{len(misses)} misses")
kit.check("three misses were filed under the over-predicted label", len(worst) == 3,
          [miss[0] for miss in worst])

## 9. The mini project's self-check (6 minutes)

The chapter's self-check for mini project 4 has three lines: the CSV has twenty rows and no empty
labels, the per-label counts add up to twenty, and the README names the label the model
over-predicts. The next cell writes the CSV as text, the way the file will look in your
repository, and the cell after it holds the one sentence you would now say to Meera, which your
README's Result opens with.

In [ ]:
buffer = io.StringIO()
writer = csv.writer(buffer, lineterminator="\n")
writer.writerow(["ticket_id", "hand_label", "model_label"])
for ticket_id, text, hand, model in TICKETS:
    writer.writerow([ticket_id, hand, model])
csv_text = buffer.getvalue()
rows = list(csv.DictReader(io.StringIO(csv_text)))
print("\n".join(csv_text.splitlines()[:4]))
print(f"and {len(rows) - 3} more rows")

In [ ]:
# TODO 16. Write the one sentence you would say to Meera: the agreement as a number, the label
# the model over-predicts, and what happens before any ticket count reaches her.
say_to_meera = __TODO16__
print(say_to_meera)

In [ ]:
kit.check("the CSV has twenty rows", len(rows) == 20, len(rows))
kit.check("no label in the CSV is empty", all(row["hand_label"] and row["model_label"] for row in rows))
kit.check("the per-label counts add up to twenty in both columns",
          sum(hand_counts.values()) == sum(model_counts.values()) == len(rows),
          f"{sum(hand_counts.values())} and {sum(model_counts.values())}")
kit.check("the sentence names the label the model over-predicts",
          bool(over) and all(label in say_to_meera for label in over), over)
kit.check("the sentence carries the agreement as a number", str(matches) in say_to_meera, matches)

The prompt below is Figure 20's four parts: the task, the allowed outputs, one example each and
the output shape. The delivery example is the chapter's own, and the refund and other examples
are written for practice and appear nowhere in the twenty tickets, since an example that is also a
test ticket flatters the agreement.

In [ ]:
FIGURE_20_PROMPT = "\n".join([
    "Label each support ticket below.",
    "Exactly one of: refund, delivery, other.",
    "Examples: 'Parcel never came' -> delivery; 'I paid for gift wrap that was never added' -> refund; "
    "'How do I reset my password?' -> other.",
    "Reply with the label only, no sentence: one line per ticket, as the ticket id and its label.",
    "",
    *[f"{ticket_id}: {text}" for ticket_id, text, hand, model in TICKETS],
])
print(FIGURE_20_PROMPT)
kit.check("the prompt lists the three allowed labels and all twenty tickets",
          "refund, delivery, other" in FIGURE_20_PROMPT
          and all(f"{ticket_id}:" in FIGURE_20_PROMPT for ticket_id, _, _, _ in TICKETS))

In [ ]:
kit.check_summary()

**Back to Meera.** The sentence printed in step 9 is the one she hears, and your README's Result
opens with it.

**Into `w00-diagnostic-llm`.** Write your own twenty tickets in `tickets.txt` and label each by
hand to a rule you write first, or keep these twenty while you learn the loop. Copy four cells
into `notebook.ipynb` in the repository: the tickets cell, the agreement loop, the per-label cell
and the self-check cell, and save the CSV text as `labels.csv`. The helper is absent there, so
turn each `kit.check(label, condition)` into `assert condition, label` and each `kit.table` into
a `print`. Then replace the practice model column with your own assistant's labels: paste the
prompt printed above into any chat assistant you already have access to, record its twenty labels
in the model column, and run the notebook again from the top. The README's Result carries the
agreement overall and per label, the label the model over-predicts, and the three worst misses
with one sentence each on why the model went wrong. Chapter 8 shows how to create the repository.